# Dense Mapped Logit-KD: Feature-KD seed-42 preflight và ETA pilot

Notebook này kiểm tra nhánh Feature KD độc lập:

\[
\mathcal{L}
=\mathcal{L}_{CE}
+0.1\,\mathcal{L}_{feature},
\qquad
\mathcal{L}_{feature}
=\operatorname{mean}
\left\|
\widehat{z}^{S}
-\operatorname{sg}(\widehat{A(z^{T})})
\right\|_2^2.
\]

- Teacher: ExpansionNet v2 frozen, feature
  `input_linear(swin_transf(image))`, `[B,144,512]`;
- căn chỉnh: pooling `12×12 → 7×7`, sau đó
  `Linear(512,256) + LayerNorm`;
- Student: S1 MobileNetV3-Large **seed 42 baseline-best**;
- preflight bắt buộc có đúng `0` optimizer step;
- pilot thời gian có đúng `100` optimizer step, sau đó khôi phục trạng thái;
- không lưu checkpoint pilot, không chạy full train, không đọc Test.

Nếu preflight thất bại, Cell 5 tự động dừng và không tạo optimizer.

## Kaggle

GPU T4 hoặc mạnh hơn, Internet Off. Gắn cùng các input đã dùng cho seed 42
Dense Mapped Logit-KD, output `Next Phase`, và chạy tuần tự từ Cell 1 đến Cell 6.


## Cell 1 — Authorization, integrity và baseline-best lineage


In [1]:
import copy, csv, gc, hashlib, json, math, os, pickle, platform, random, shutil, subprocess, sys, time
from argparse import Namespace
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import torch

STARTED = time.perf_counter()
SEED = 42
INPUT_ROOT = Path(os.environ.get("KLCN_INPUT_ROOT", "/kaggle/input"))
NOTEBOOK_ROOT = (
    INPUT_ROOT / "notebooks"
    if (INPUT_ROOT / "notebooks").is_dir()
    else INPUT_ROOT
)
DATASET_ROOT = (
    INPUT_ROOT / "datasets"
    if (INPUT_ROOT / "datasets").is_dir()
    else INPUT_ROOT
)
OUT = Path(os.environ.get(
    "KLCN_OUTPUT_ROOT",
    "/kaggle/working//XLKD05_Dense_Mapped_LogitKD_feature_kd_seed42_pilot",
))
AUDIT = OUT / "audits"
CONFIG = OUT / "configs"
LOG = OUT / "logs"
RESULT = OUT / "results"
CHECKPOINT = OUT / "checkpoints"
MANIFEST = OUT / "manifests"
for directory in [OUT, AUDIT, CONFIG, LOG, RESULT, CHECKPOINT, MANIFEST]:
    directory.mkdir(parents=True, exist_ok=True)

def now():
    return datetime.now(timezone.utc).isoformat()

def read_json(path):
    with Path(path).open("r", encoding="utf-8") as handle:
        return json.load(handle)

def write_json(path, payload):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(
        json.dumps(payload, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )
    os.replace(temporary, path)

def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(8 * 1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def resolve_inside(root, relative_path):
    root = Path(root).resolve()
    target = (root / relative_path).resolve()
    assert target.is_relative_to(root), (root, relative_path, target)
    return target

_DISCOVERY_INDEX = {}
for _candidate in NOTEBOOK_ROOT.rglob("*"):
    if _candidate.is_file():
        _DISCOVERY_INDEX.setdefault(_candidate.name, []).append(
            _candidate.resolve()
        )

def exactly_one(name):
    matches = sorted(_DISCOVERY_INDEX.get(name, []))
    assert len(matches) == 1, (
        f"Cần đúng một {name} dưới {NOTEBOOK_ROOT}; "
        f"tìm thấy {len(matches)}: {matches}"
    )
    return matches[0]

def verify_csv_manifest(root, reference):
    path = resolve_inside(root, reference["relative_path"])
    assert path.is_file() and sha256(path) == reference["sha256"]
    with path.open("r", encoding="utf-8", newline="") as handle:
        rows = list(csv.DictReader(handle))
    if "rows" in reference:
        assert len(rows) == int(reference["rows"])
    for row in rows:
        artifact = resolve_inside(root, row["relative_path"])
        assert artifact.is_file()
        assert artifact.stat().st_size == int(row["size_bytes"])
        assert sha256(artifact) == row["sha256"]
    return rows

def set_deterministic(seed):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
    torch.use_deterministic_algorithms(True, warn_only=False)
    torch.backends.cuda.enable_flash_sdp(False)
    torch.backends.cuda.enable_mem_efficient_sdp(False)
    torch.backends.cuda.enable_math_sdp(True)

def find_expansionnet_manifest():
    preferred = INPUT_ROOT / "models"
    search_root = preferred if preferred.is_dir() else INPUT_ROOT
    matches = []
    for candidate in search_root.rglob("MODEL_SOURCE_MANIFEST.json"):
        try:
            payload = read_json(candidate)
        except Exception:
            continue
        if payload.get("model_id") == "expansionnet_v2":
            matches.append(candidate.resolve())
    assert len(matches) == 1, matches
    return matches[0]

if not torch.cuda.is_available():
    raise RuntimeError(
        "XLKD-04 yêu cầu GPU T4 hoặc mạnh hơn. Hãy bật Accelerator rồi chạy lại."
    )
set_deterministic(SEED)
device = torch.device("cuda")

X02_PATH = exactly_one("XLKD02_DENSE_MAPPED_LOSS_PREFLIGHT_COMPLETE.json")
AUTH_PATH = exactly_one(
    "XLKD03_DENSE_MAPPED_LOGIT_KD_TRAINING_READINESS_AUTHORIZED.json"
)
X00_AUTH_PATH = exactly_one(
    "XLKD01_GPU_PROBABILITY_MASS_PREFLIGHT_AUTHORIZED.json"
)
P1_PATH = exactly_one("PHASE1_FORMAL_DELIVERABLES_READY.json")
P2A_PATH = exactly_one("PHASE2_SHARED_TRAINING_BUNDLE_READY.json")
S1_PATH = exactly_one("S1_SEED42_FULL_RUN_COMPLETE.json")
P3B_PATH = exactly_one(
    "PHASE3_TEACHER_A_EXPANSIONNET_V2_PREFLIGHT_READY.json"
)
P3E_PATH = exactly_one(
    "PHASE3_TEACHER_A_EXPANSIONNET_V2_HANDOFF_COMPLETE.json"
)
MODEL_MANIFEST_PATH = find_expansionnet_manifest()
MODEL_ROOT = MODEL_MANIFEST_PATH.parent

X02, AUTH, X00_AUTH, P1, P2A, S1, P3B, P3E = map(
    read_json,
    [X02_PATH, AUTH_PATH, X00_AUTH_PATH, P1_PATH, P2A_PATH,
     S1_PATH, P3B_PATH, P3E_PATH],
)
assert X02["status"] == "complete" and X02["overall_gate"] is True
assert X02["decision"] == "authorize_xlkd03_training_readiness"
assert X02["full_training_authorized"] is False
assert AUTH["status"] == "authorized"
assert AUTH["required_next"].startswith("XLKD-03")
assert AUTH["full_training_authorized"] is False
assert float(AUTH["temperature"]) == 4.0
assert float(AUTH["alpha_human_ce"]) == 0.5
assert float(AUTH["alpha_kd"]) == 0.5
assert int(AUTH["physical_batch_size"]) == 1
assert X00_AUTH["status"] == "authorized"
assert P1["status"] == "ready"
assert P2A["status"] == "ready"
assert S1["status"] == "complete"
assert S1["student_id"] == "S1" and int(S1["seed"]) == 42
assert P3B["status"] == "ready" and P3B["teacher"] == "ExpansionNet v2"
assert P3E["status"] == "complete"
for marker in [X02, AUTH, X00_AUTH, S1, P3B, P3E]:
    assert marker["test_set_status"] == "sealed"
    assert marker["test_set_used"] is False
assert P1["test_set_status"] == "sealed" and P1["test_used"] is False
assert P2A["data_policy"]["test_set_status"] == "sealed"
assert P2A["data_policy"]["test_set_used"] is False

x02_rows = verify_csv_manifest(X02_PATH.parent, X02["manifest"]["csv"])
assert len(x02_rows) == 11
p2a_rows = verify_csv_manifest(P2A_PATH.parent, P2A["manifests"]["csv"])
assert len(p2a_rows) == 38

decision = read_json(X02_PATH.parent / "results" / "xlkd02_decision.json")
assert decision["overall_gate"] is True
assert all(decision["gates"].values())
LOSS_CONTRACT_PATH = (
    X02_PATH.parent / "configs" / "xlkd02_executed_loss_contract.json"
).resolve()
LOSS_CONTRACT = read_json(LOSS_CONTRACT_PATH)
assert LOSS_CONTRACT["status"] == "locked"
assert LOSS_CONTRACT["temperature"] == 4.0
assert LOSS_CONTRACT["full_training_authorized"] is False

mapping_ref = X00_AUTH["mapping_contract"]
MAPPING_PATH = resolve_inside(X00_AUTH_PATH.parent, mapping_ref["relative_path"])
assert sha256(MAPPING_PATH) == mapping_ref["sha256"]
MAPPING_PAYLOAD = read_json(MAPPING_PATH)
mapping = {
    int(source): int(target)
    for source, target in MAPPING_PAYLOAD["mapping"].items()
}
assert len(mapping) == 9959
assert MAPPING_PAYLOAD["mapping_collisions"] == 0

model_manifest = read_json(MODEL_MANIFEST_PATH)
assert model_manifest["model_id"] == "expansionnet_v2"

def verify_model_record(record):
    target = resolve_inside(MODEL_ROOT, record["relative_path"])
    assert target.is_file()
    assert target.stat().st_size == int(record["size_bytes"])
    assert sha256(target) == record["sha256"]
    return target

CHECKPOINT_PATH = verify_model_record(model_manifest["checkpoint"])
NATIVE_VOCAB_PATH = verify_model_record(model_manifest["vocabulary"])
for record in model_manifest["required_files"]:
    verify_model_record(record)
for record in model_manifest["integration_patches"]:
    target = resolve_inside(MODEL_ROOT, record["file"])
    assert target.is_file() and sha256(target) == record["sha256"]

with NATIVE_VOCAB_PATH.open("rb") as handle:
    native_tokens = pickle.load(handle)
NATIVE_WORD2IDX = native_tokens["word2idx_dict"]
NATIVE_IDX2WORD = native_tokens["idx2word_list"]
NATIVE_BOS = int(NATIVE_WORD2IDX[native_tokens["sos_str"]])
NATIVE_EOS = int(NATIVE_WORD2IDX[native_tokens["eos_str"]])
NATIVE_PAD = int(NATIVE_WORD2IDX["PAD"])
NATIVE_UNK = int(NATIVE_WORD2IDX["UNK"])
assert (NATIVE_PAD, NATIVE_BOS, NATIVE_EOS, NATIVE_UNK) == (78, 79, 77, 80)

PSEUDO_PATH = (
    P3E_PATH.parent / "pseudocaptions"
    / "teacher_a_expansionnet_v2_train_restval.jsonl"
).resolve()
PSEUDO_SUMMARY_PATH = (
    P3E_PATH.parent / "pseudocaptions"
    / "teacher_a_expansionnet_v2_pseudocaption_summary.json"
).resolve()
PSEUDO_SUMMARY = read_json(PSEUDO_SUMMARY_PATH)
assert PSEUDO_SUMMARY["records"] == 113287
assert sha256(PSEUDO_PATH) == PSEUDO_SUMMARY["jsonl_sha256"]
assert PSEUDO_SUMMARY["validation_overlap"] == 0
assert PSEUDO_SUMMARY["held_out_test_overlap"] == 0

BASELINE_CHECKPOINT = exactly_one("s1_mobilenetv3_seed42_best.pt")
assert sha256(BASELINE_CHECKPOINT) == S1["best_checkpoint"]["sha256"]
DECODER_CONTRACT = exactly_one("shared_transformer_decoder_contract.json")

write_json(AUDIT / "xlkd03_cell01_authorization.json", {
    "schema_version": "xlkd03_cell01_v1",
    "created_at_utc": now(),
    "status": "passed",
    "xlkd02_manifest_rows": len(x02_rows),
    "phase2a_manifest_rows": len(p2a_rows),
    "xlkd02_gates": decision["gates"],
    "teacher_checkpoint_sha256": sha256(CHECKPOINT_PATH),
    "student_checkpoint_sha256": sha256(BASELINE_CHECKPOINT),
    "mapping_tokens": len(mapping),
    "pseudo_caption_rows": 113287,
    "gpu": torch.cuda.get_device_name(0),
    "python": platform.python_version(),
    "pytorch": torch.__version__,
    "strict_deterministic": True,
    "checkpoint_payload_read": False,
    "dataset_read": False,
    "validation_read": False,
    "full_training_started": False,
    "test_set_status": "sealed",
    "test_set_used": False,
    "elapsed_seconds": time.perf_counter() - STARTED,
})
print("=" * 78)
print("XLKD-03 CELL 1 — AUTHORIZATION AND INTEGRITY")
print("=" * 78)
print("Status                    : PASSED")
print("XLKD-02 manifest          : 11 / 11 VERIFIED")
print("Phase 02A manifest        : 38 / 38 VERIFIED")
print("XLKD-02 gates             : 9 / 9 PASSED")
print("Vocabulary mapping        : 9,959 VERIFIED")
print("Pseudo-caption corpus     : 113,287 VERIFIED")
print("Teacher / Student ckpt    : VERIFIED / VERIFIED")
print("Validation / Test         : NOT READ / SEALED")
print("Full training             : NOT STARTED")
print("Progress                  : 1 / 7 code cells")


# XLKD-04 may start only from the exact, verified XLKD-03 output.
X03_COMPLETE_PATH = exactly_one(
    "XLKD03_DENSE_MAPPED_TRAINING_READINESS_COMPLETE.json"
)
X04_AUTH_PATH = exactly_one(
    "XLKD04_DENSE_MAPPED_LOGIT_KD_SEED42_FULL_RUN_AUTHORIZED.json"
)
X03_COMPLETE = read_json(X03_COMPLETE_PATH)
X04_AUTH = read_json(X04_AUTH_PATH)
assert X03_COMPLETE["status"] == "complete"
assert X03_COMPLETE["production_data_pipeline"] == "passed"
assert X03_COMPLETE["model_optimizer_engine"] == "passed"
assert X03_COMPLETE["validation_policy_inheritance"] == "passed"
assert X03_COMPLETE["discarded_pilot"] == "passed"
assert X03_COMPLETE["production_contract"] == "locked"
assert X03_COMPLETE["xlkd04_seed42_authorized"] is True
assert X04_AUTH["status"] == "authorized"
assert int(X04_AUTH["seed"]) == 42
assert X04_AUTH["full_training_authorized"] is True
assert X04_AUTH["validation_selection_authorized"] is True
assert X04_AUTH["confirmation_seeds_authorized"] is False
assert X04_AUTH["test_set_status"] == "sealed"
assert X04_AUTH["test_set_used"] is False

x03_rows = verify_csv_manifest(
    X03_COMPLETE_PATH.parent, X03_COMPLETE["manifest"]["csv"]
)
assert len(x03_rows) == 12
contract_ref = X04_AUTH["production_contract"]
PRODUCTION_CONTRACT_PATH = resolve_inside(
    X04_AUTH_PATH.parent, contract_ref["relative_path"]
)
assert sha256(PRODUCTION_CONTRACT_PATH) == contract_ref["sha256"]
PRODUCTION_CONTRACT = read_json(PRODUCTION_CONTRACT_PATH)
assert PRODUCTION_CONTRACT["status"] == "locked"
assert PRODUCTION_CONTRACT["loss"]["temperature"] == 4.0
assert PRODUCTION_CONTRACT["loss"]["alpha_human_ce"] == 0.5
assert PRODUCTION_CONTRACT["loss"]["alpha_kd"] == 0.5
assert PRODUCTION_CONTRACT["training"]["physical_batch"] == 1
assert PRODUCTION_CONTRACT["training"]["gradient_accumulation"] == 64
assert PRODUCTION_CONTRACT["validation"]["selection_primary"] == "CIDEr-D"
assert PRODUCTION_CONTRACT["test_set_status"] == "sealed"
assert PRODUCTION_CONTRACT["test_set_used"] is False

roadmap_ref = X04_AUTH["roadmap"]
FINAL_ROADMAP_PATH = resolve_inside(
    X04_AUTH_PATH.parent, roadmap_ref["relative_path"]
)
assert sha256(FINAL_ROADMAP_PATH) == roadmap_ref["sha256"]

resume_matches = sorted(
    _DISCOVERY_INDEX.get("xlkd04_dense_mapped_seed42_last.pt", [])
)
assert len(resume_matches) <= 1, resume_matches
INPUT_RESUME_CHECKPOINT = (
    resume_matches[0] if resume_matches else None
)

# Compatibility names used by the already-audited Validation/full-run engine.
OUTPUT_ROOT = OUT
AUDIT_DIR = AUDIT
CONFIG_DIR = CONFIG
LOG_DIR = LOG
RESULT_DIR = RESULT
CHECKPOINT_DIR = CHECKPOINT
MANIFEST_DIR = MANIFEST
VALIDATION_DIR = OUT / "validation"
for directory in [VALIDATION_DIR]:
    directory.mkdir(parents=True, exist_ok=True)
now_utc = now
sha256_file = sha256
PSEUDO_CORPUS = PSEUDO_PATH

BASELINE_CIDER = float(S1["best_validation_metrics"]["CIDEr-D"])
BASELINE_BLEU4 = float(S1["best_validation_metrics"]["BLEU-4"])
BASELINE_CE = float(S1["best_validation_cross_entropy"])
BASELINE_METRICS = {
    key: float(value)
    for key, value in S1["best_validation_metrics"].items()
}

write_json(AUDIT / "feature_kd_dense_lineage_cell01.json", {
    "schema_version": "xlkd04_cell01_v1",
    "created_at_utc": now(),
    "status": "passed",
    "xlkd03_manifest_rows": len(x03_rows),
    "production_contract_sha256": sha256(PRODUCTION_CONTRACT_PATH),
    "roadmap_sha256": sha256(FINAL_ROADMAP_PATH),
    "seed": 42,
    "resume_checkpoint_found": INPUT_RESUME_CHECKPOINT is not None,
    "validation_authorized": True,
    "full_training_authorized": True,
    "confirmation_seeds_authorized": False,
    "test_set_status": "sealed",
    "test_set_used": False,
})
print("=" * 78)
print("FEATURE-KD CELL 1 — DENSE LINEAGE VERIFICATION")
print("=" * 78)
print("Status                    : PASSED")
print("XLKD-03 manifest          : 12 / 12 VERIFIED")
print("Production contract       : LOCKED")
print("Seed                      : 42")
print("Resume checkpoint found   :", INPUT_RESUME_CHECKPOINT is not None)
print("Validation                : AUTHORIZED")
print("Dense lineage training    : PREVIOUSLY AUTHORIZED")
print("Confirmation seeds        : NOT YET")
print("Test Set status           : SEALED")


# Current next-phase authorization.  This is independent from the historical
# XLKD-04 full-run authorization verified above.
NEXT_PHASE_PATH = exactly_one(
    "XLKD05_Dense_Mapped_LogitKD_NEXT_PHASE_ENTRY_COMPLETE.json"
)
FEATURE_AUTH_PATH = exactly_one("FEATURE_KD_GPU_PREFLIGHT_AUTHORIZED.json")
NEXT_PHASE = read_json(NEXT_PHASE_PATH)
FEATURE_AUTH = read_json(FEATURE_AUTH_PATH)
assert NEXT_PHASE["status"] == "complete"
assert NEXT_PHASE["reproduction_gate"] == "passed"
assert NEXT_PHASE["final_scientific_gate"] == "pending"
assert NEXT_PHASE["test_set_status"] == "sealed"
assert NEXT_PHASE["test_set_used"] is False
assert FEATURE_AUTH["status"] == "authorized"
assert int(FEATURE_AUTH["optimizer_steps_allowed"]) == 0
assert FEATURE_AUTH["test_set_status"] == "sealed"
assert FEATURE_AUTH["test_set_used"] is False

feature_ref = FEATURE_AUTH["contract"]
FEATURE_CONTRACT_PATH = resolve_inside(
    FEATURE_AUTH_PATH.parent, feature_ref["relative_path"]
)
assert sha256(FEATURE_CONTRACT_PATH) == feature_ref["sha256"]
FEATURE_CONTRACT = read_json(FEATURE_CONTRACT_PATH)
assert FEATURE_CONTRACT["status"] == "locked"
assert FEATURE_CONTRACT["teacher"]["feature_tap"] == (
    "input_linear(swin_transf(image))"
)
assert FEATURE_CONTRACT["teacher"]["expected_shape"] == ["B", 144, 512]
assert FEATURE_CONTRACT["student"]["expected_shape"] == ["B", 49, 256]
assert FEATURE_CONTRACT["alignment"]["adapter"] == (
    "Linear(512,256) + LayerNorm"
)
assert FEATURE_CONTRACT["alignment"]["identity_projection_forbidden"] is True
assert float(FEATURE_CONTRACT["loss"]["lambda_primary"]) == 0.1
assert FEATURE_CONTRACT["preflight_gates"]["optimizer_steps"] == 0

write_json(AUDIT / "feature_kd_cell01_authorization.json", {
    "schema_version": "feature_kd_seed42_cell01_v1",
    "created_at_utc": now(),
    "status": "passed",
    "seed": SEED,
    "student_initialization": "S1 seed-42 baseline-best",
    "feature_contract_sha256": sha256(FEATURE_CONTRACT_PATH),
    "preflight_optimizer_steps_allowed": 0,
    "timing_pilot_optimizer_steps": 100,
    "timing_pilot_weights_persisted": False,
    "full_training_started": False,
    "test_set_status": "sealed",
    "test_set_used": False,
})
print("Feature-KD authorization  : VERIFIED")
print("Student initialization    : S1 seed-42 BASELINE-BEST")
print("Preflight optimizer steps : 0")
print("Discarded timing pilot    : 100 steps after preflight only")


XLKD-03 CELL 1 — AUTHORIZATION AND INTEGRITY
Status                    : PASSED
XLKD-02 manifest          : 11 / 11 VERIFIED
Phase 02A manifest        : 38 / 38 VERIFIED
XLKD-02 gates             : 9 / 9 PASSED
Vocabulary mapping        : 9,959 VERIFIED
Pseudo-caption corpus     : 113,287 VERIFIED
Teacher / Student ckpt    : VERIFIED / VERIFIED
Validation / Test         : NOT READ / SEALED
Full training             : NOT STARTED
Progress                  : 1 / 7 code cells
FEATURE-KD CELL 1 — DENSE LINEAGE VERIFICATION
Status                    : PASSED
XLKD-03 manifest          : 12 / 12 VERIFIED
Production contract       : LOCKED
Seed                      : 42
Resume checkpoint found   : False
Validation                : AUTHORIZED
Dense lineage training    : PREVIOUSLY AUTHORIZED
Confirmation seeds        : NOT YET
Test Set status           : SEALED
Feature-KD authorization  : VERIFIED
Student initialization    : S1 seed-42 BASELINE-BEST
Preflight optimizer steps : 0
Discarded timin

## Cell 2 — Production Train+Restval pipeline


In [2]:
STARTED = time.perf_counter()
from collections import defaultdict

import pyarrow.parquet as pq
from PIL import Image
from torch.utils.data import DataLoader, Dataset, Sampler
from torchvision.transforms import InterpolationMode
from torchvision.transforms import RandomResizedCrop
from torchvision.transforms import functional as TF

TRAIN_PARQUET = exactly_one("mscoco_train.parquet")
IMAGE_MANIFEST_PATH = exactly_one("mscoco_image_manifest.parquet")

def find_image_leaf(name, prefix):
    matches = []
    for path in DATASET_ROOT.rglob(name):
        if path.is_dir() and next(path.glob(f"{prefix}*.jpg"), None):
            matches.append(path.resolve())
    matches = sorted(set(matches))
    assert len(matches) == 1, matches
    return matches[0]

TRAIN_LEAF = find_image_leaf("train2014", "COCO_train2014_")
VAL_LEAF = find_image_leaf("val2014", "COCO_val2014_")

path_table = pq.read_table(
    IMAGE_MANIFEST_PATH,
    columns=["image_id", "experiment_split", "source_partition", "image_path"],
    filters=[("experiment_split", "=", "train")],
)
TRAIN_PATHS = {}
for row in path_table.to_pylist():
    image_id = int(row["image_id"])
    leaf = TRAIN_LEAF if row["source_partition"] == "train2014" else VAL_LEAF
    TRAIN_PATHS[image_id] = (leaf / Path(row["image_path"]).name).resolve()
assert len(TRAIN_PATHS) == 113287

TRAIN_CAPTIONS = defaultdict(list)
for arrow_batch in pq.ParquetFile(TRAIN_PARQUET).iter_batches(
    batch_size=16384,
    columns=["annotation_id", "image_id", "token_ids"],
):
    for row in arrow_batch.to_pylist():
        TRAIN_CAPTIONS[int(row["image_id"])].append((
            int(row["annotation_id"]),
            tuple(int(value) for value in row["token_ids"]),
        ))
for image_id in TRAIN_CAPTIONS:
    TRAIN_CAPTIONS[image_id].sort()
assert len(TRAIN_CAPTIONS) == 113287

PSEUDO_BY_IMAGE = {}
with PSEUDO_PATH.open("r", encoding="utf-8") as handle:
    for line in handle:
        row = json.loads(line)
        image_id = int(row["image_id"])
        teacher_ids = tuple(int(value) for value in row["teacher_token_ids"])
        student_ids = tuple(int(value) for value in row["student_token_ids"])
        assert len(teacher_ids) == len(student_ids)
        assert [mapping[value] for value in teacher_ids] == list(student_ids)
        assert image_id not in PSEUDO_BY_IMAGE
        PSEUDO_BY_IMAGE[image_id] = (teacher_ids, student_ids)
assert set(PSEUDO_BY_IMAGE) == set(TRAIN_PATHS)
TRAIN_IMAGE_IDS = tuple(sorted(TRAIN_PATHS))

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

def sample_seed(epoch, image_id):
    text = f"{SEED}|{epoch}|{image_id}|xlkd04"
    return int.from_bytes(
        hashlib.sha256(text.encode("utf-8")).digest()[:8], "big"
    ) % (2 ** 63 - 1)

def dual_train_transform(image, epoch, image_id):
    with torch.random.fork_rng(devices=[]):
        torch.manual_seed(sample_seed(epoch, image_id))
        top, left, height, width = RandomResizedCrop.get_params(
            image, scale=(0.80, 1.00), ratio=(0.75, 4 / 3)
        )
        do_flip = bool(torch.rand(()) < 0.5)
    teacher = TF.resized_crop(
        image, top, left, height, width, [384, 384],
        InterpolationMode.BILINEAR, antialias=True,
    )
    student = TF.resized_crop(
        image, top, left, height, width, [224, 224],
        InterpolationMode.BICUBIC, antialias=True,
    )
    if do_flip:
        teacher, student = TF.hflip(teacher), TF.hflip(student)
    teacher = TF.normalize(TF.to_tensor(teacher), IMAGENET_MEAN, IMAGENET_STD)
    student = TF.normalize(TF.to_tensor(student), IMAGENET_MEAN, IMAGENET_STD)
    return teacher, student

class ProductionTrainDataset(Dataset):
    def __init__(self):
        self.epoch = 0

    def set_epoch(self, epoch):
        self.epoch = int(epoch)

    def __len__(self):
        return len(TRAIN_IMAGE_IDS)

    def __getitem__(self, index):
        image_id = TRAIN_IMAGE_IDS[index]
        captions = TRAIN_CAPTIONS[image_id]
        offset = image_id % len(captions)
        annotation_id, human_ids = captions[
            (self.epoch + offset) % len(captions)
        ]
        teacher_ids, student_ids = PSEUDO_BY_IMAGE[image_id]
        with Image.open(TRAIN_PATHS[image_id]) as source:
            teacher_image, student_image = dual_train_transform(
                source.convert("RGB"), self.epoch, image_id
            )
        return {
            "image_id": image_id,
            "annotation_id": annotation_id,
            "teacher_image": teacher_image,
            "student_image": student_image,
            "teacher_ids": torch.tensor(teacher_ids, dtype=torch.long),
            "student_pseudo_ids": torch.tensor(student_ids, dtype=torch.long),
            "human_ids": torch.tensor(human_ids, dtype=torch.long),
        }

class EpochCursorSampler(Sampler):
    def __init__(self, length, epoch=0, cursor=0):
        self.length = int(length)
        self.epoch = int(epoch)
        self.cursor = int(cursor)

    def __iter__(self):
        generator = torch.Generator().manual_seed(
            SEED + self.epoch * 1000003
        )
        order = torch.randperm(self.length, generator=generator).tolist()
        return iter(order[self.cursor:])

    def __len__(self):
        return self.length - self.cursor

def production_collate(rows):
    assert len(rows) == 1
    row = rows[0]
    return {
        "image_ids": torch.tensor([row["image_id"]], dtype=torch.long),
        "annotation_ids": torch.tensor([row["annotation_id"]], dtype=torch.long),
        "teacher_image": row["teacher_image"].unsqueeze(0),
        "student_image": row["student_image"].unsqueeze(0),
        "teacher_ids": row["teacher_ids"].unsqueeze(0),
        "student_pseudo_ids": row["student_pseudo_ids"].unsqueeze(0),
        "human_ids": row["human_ids"].unsqueeze(0),
    }

TRAIN_DATASET = ProductionTrainDataset()

def build_train_loader(epoch, cursor=0):
    TRAIN_DATASET.set_epoch(epoch)
    return DataLoader(
        TRAIN_DATASET,
        batch_size=1,
        sampler=EpochCursorSampler(len(TRAIN_DATASET), epoch, cursor),
        num_workers=2,
        pin_memory=True,
        persistent_workers=False,
        collate_fn=production_collate,
    )

batch_a = next(iter(build_train_loader(0)))
batch_b = next(iter(build_train_loader(0)))
for key in [
    "image_ids", "annotation_ids", "teacher_image", "student_image",
    "teacher_ids", "student_pseudo_ids", "human_ids",
]:
    assert torch.equal(batch_a[key], batch_b[key])
assert batch_a["teacher_image"].shape == (1, 3, 384, 384)
assert batch_a["student_image"].shape == (1, 3, 224, 224)
assert torch.isfinite(batch_a["teacher_image"]).all()
assert torch.isfinite(batch_a["student_image"]).all()

write_json(AUDIT / "feature_kd_cell02_data_pipeline.json", {
    "schema_version": "xlkd04_cell02_v1",
    "created_at_utc": now(),
    "status": "passed",
    "train_images_per_epoch": len(TRAIN_DATASET),
    "human_caption_sampling": "one per image per epoch, cyclic",
    "teacher_sequence_per_image": 1,
    "physical_batch_size": 1,
    "shared_stateless_geometry": True,
    "teacher_resolution": [384, 384],
    "student_resolution": [224, 224],
    "teacher_student_sequence_alignment": "113287 / 113287",
    "two_loader_first_batch_exact": True,
    "resume_cursor_supported": True,
    "validation_read": False,
    "test_parquet_opened": False,
    "test_images_read": False,
    "test_set_status": "sealed",
    "test_set_used": False,
    "elapsed_seconds": time.perf_counter() - STARTED,
})
print("=" * 78)
print("XLKD-03 CELL 2 — PRODUCTION DATA PIPELINE")
print("=" * 78)
print("Status                    : PASSED")
print("Train images / epoch      : 113,287")
print("Human caption / image     : 1 CYCLIC")
print("Teacher sequence / image  : 1 FIXED")
print("Sequence alignment        : 113,287 / 113,287")
print("Physical batch            : 1")
print("Teacher / Student input   : 384 / 224")
print("Stateless shared geometry : True")
print("Validation / Test         : NOT READ / SEALED")
print("Progress                  : 2 / 7 code cells")


VALIDATION_PARQUET = exactly_one("mscoco_validation.parquet")
validation_path_table = pq.read_table(
    IMAGE_MANIFEST_PATH,
    columns=["image_id", "experiment_split", "source_partition", "image_path"],
    filters=[("experiment_split", "=", "validation")],
)
VALIDATION_PATHS = {}
for row in validation_path_table.to_pylist():
    image_id = int(row["image_id"])
    leaf = TRAIN_LEAF if row["source_partition"] == "train2014" else VAL_LEAF
    VALIDATION_PATHS[image_id] = (
        leaf / Path(row["image_path"]).name
    ).resolve()
assert len(VALIDATION_PATHS) == 5000

VALIDATION_RECORDS = []
VALIDATION_REFERENCES = defaultdict(list)
for arrow_batch in pq.ParquetFile(VALIDATION_PARQUET).iter_batches(
    batch_size=8192,
    columns=["annotation_id", "image_id", "token_ids", "caption"],
):
    for row in arrow_batch.to_pylist():
        image_id = int(row["image_id"])
        VALIDATION_RECORDS.append((
            int(row["annotation_id"]),
            image_id,
            tuple(int(value) for value in row["token_ids"]),
        ))
        VALIDATION_REFERENCES[image_id].append(row["caption"])
assert len(VALIDATION_RECORDS) == 25010
assert len(VALIDATION_REFERENCES) == 5000
VALIDATION_IMAGE_IDS = tuple(sorted(VALIDATION_PATHS))

def validation_transform(image):
    image = TF.resize(
        image, 256, interpolation=InterpolationMode.BICUBIC,
        antialias=True,
    )
    tensor = TF.to_tensor(TF.center_crop(image, [224, 224]))
    return TF.normalize(tensor, IMAGENET_MEAN, IMAGENET_STD)

class ValidationCaptionDataset(Dataset):
    def __len__(self):
        return len(VALIDATION_RECORDS)

    def __getitem__(self, index):
        annotation_id, image_id, token_ids = VALIDATION_RECORDS[index]
        with Image.open(VALIDATION_PATHS[image_id]) as source:
            image = validation_transform(source.convert("RGB"))
        return {
            "image_id": image_id,
            "annotation_id": annotation_id,
            "student_image": image,
            "token_ids": torch.tensor(token_ids, dtype=torch.long),
        }

class ValidationImageDataset(Dataset):
    def __len__(self):
        return len(VALIDATION_IMAGE_IDS)

    def __getitem__(self, index):
        image_id = VALIDATION_IMAGE_IDS[index]
        with Image.open(VALIDATION_PATHS[image_id]) as source:
            image = validation_transform(source.convert("RGB"))
        return {"image_id": image_id, "student_image": image}

def pad_sequences(sequences):
    maximum = max(item.numel() for item in sequences)
    result = torch.zeros(len(sequences), maximum, dtype=torch.long)
    for index, item in enumerate(sequences):
        result[index, :item.numel()] = item
    return result

def validation_caption_collate(rows):
    full = pad_sequences([row["token_ids"] for row in rows])
    return {
        "image_ids": torch.tensor(
            [row["image_id"] for row in rows], dtype=torch.long
        ),
        "annotation_ids": torch.tensor(
            [row["annotation_id"] for row in rows], dtype=torch.long
        ),
        "student_images": torch.stack(
            [row["student_image"] for row in rows]
        ),
        "decoder_input_ids": full[:, :-1],
        "labels": full[:, 1:],
        "decoder_attention_mask": full[:, :-1].ne(0),
    }

def image_collate(rows):
    return {
        "image_ids": torch.tensor(
            [row["image_id"] for row in rows], dtype=torch.long
        ),
        "student_images": torch.stack(
            [row["student_image"] for row in rows]
        ),
    }

VALIDATION_CAPTION_DATASET = ValidationCaptionDataset()
VALIDATION_IMAGE_DATASET = ValidationImageDataset()
VALIDATION_CAPTION_LOADER = DataLoader(
    VALIDATION_CAPTION_DATASET,
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    collate_fn=validation_caption_collate,
)
VALIDATION_IMAGE_LOADER = DataLoader(
    VALIDATION_IMAGE_DATASET,
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    collate_fn=image_collate,
)

assert set(VALIDATION_PATHS).isdisjoint(TRAIN_PATHS)
write_json(AUDIT / "feature_kd_cell02_validation_data.json", {
    "schema_version": "xlkd04_cell02_validation_data_v1",
    "created_at_utc": now(),
    "status": "passed",
    "training_images": len(TRAIN_DATASET),
    "validation_images": len(VALIDATION_IMAGE_DATASET),
    "validation_captions": len(VALIDATION_CAPTION_DATASET),
    "validation_overlap_with_training": 0,
    "test_parquet_opened": False,
    "test_images_read": False,
    "test_set_status": "sealed",
    "test_set_used": False,
})
print("Validation images         : 5,000")
print("Validation references     : 25,010")
print("Train/Validation overlap  : 0")
print("Test Set status           : SEALED")


XLKD-03 CELL 2 — PRODUCTION DATA PIPELINE
Status                    : PASSED
Train images / epoch      : 113,287
Human caption / image     : 1 CYCLIC
Teacher sequence / image  : 1 FIXED
Sequence alignment        : 113,287 / 113,287
Physical batch            : 1
Teacher / Student input   : 384 / 224
Stateless shared geometry : True
Validation / Test         : NOT READ / SEALED
Progress                  : 2 / 7 code cells
Validation images         : 5,000
Validation references     : 25,010
Train/Validation overlap  : 0
Test Set status           : SEALED


## Cell 3 — Feature alignment và pilot optimizer engine


In [3]:
STARTED = time.perf_counter()
import torch.nn as nn
import torch.nn.functional as F
from torch.torch_version import TorchVersion
from torchvision.models import mobilenet_v3_large

SOURCE_ROOT = MODEL_ROOT / "ExpansionNet_v2_src"
assert SOURCE_ROOT.is_dir()
if str(SOURCE_ROOT) not in sys.path:
    sys.path.insert(0, str(SOURCE_ROOT))
from models.End_ExpansionNet_v2 import End_ExpansionNet_v2

drop_args = Namespace(
    enc=0.0, dec=0.0, enc_input=0.0, dec_input=0.0, other=0.0
)
TEACHER = End_ExpansionNet_v2(
    swin_img_size=384, swin_patch_size=4, swin_in_chans=3,
    swin_embed_dim=192, swin_depths=[2, 2, 18, 2],
    swin_num_heads=[6, 12, 24, 48], swin_window_size=12,
    swin_mlp_ratio=4.0, swin_qkv_bias=True, swin_qk_scale=None,
    swin_drop_rate=0.0, swin_attn_drop_rate=0.0,
    swin_drop_path_rate=0.0, swin_norm_layer=torch.nn.LayerNorm,
    swin_ape=False, swin_patch_norm=True, swin_use_checkpoint=False,
    final_swin_dim=1536, d_model=512, N_enc=3, N_dec=3,
    num_heads=8, ff=2048, num_exp_enc_list=[32, 64, 128, 256, 512],
    num_exp_dec=16, output_word2idx=NATIVE_WORD2IDX,
    output_idx2word=NATIVE_IDX2WORD, max_seq_len=74,
    drop_args=drop_args, rank=device,
)
teacher_payload = torch.load(
    CHECKPOINT_PATH, map_location="cpu", weights_only=False
)
incompatible = TEACHER.load_state_dict(
    teacher_payload["model_state_dict"], strict=True
)
assert not incompatible.missing_keys and not incompatible.unexpected_keys
del teacher_payload
gc.collect()
TEACHER = TEACHER.to(device).eval()
for parameter in TEACHER.parameters():
    parameter.requires_grad_(False)
assert all(not parameter.requires_grad for parameter in TEACHER.parameters())

DECODER_CONTRACT = exactly_one("shared_transformer_decoder_contract.json")
decoder_config = read_json(DECODER_CONTRACT)["architecture"]
assert int(decoder_config["vocabulary_size"]) == 10080

class S1Encoder(nn.Module):
    def __init__(self):
        super().__init__()
        model = mobilenet_v3_large(weights=None)
        self.backbone = model.features
        self.projection = nn.Linear(960, 256)
        self.visual_position_embedding = nn.Parameter(
            torch.empty(1, 49, 256)
        )
        self.output_norm = nn.LayerNorm(256)

    def forward(self, images):
        feature = self.backbone(images)
        tokens = feature.flatten(2).transpose(1, 2).contiguous()
        assert tokens.shape[1:] == (49, 960)
        return self.output_norm(
            self.projection(tokens) + self.visual_position_embedding
        )

class SharedDecoder(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.vocab = int(config["vocabulary_size"])
        self.d_model = int(config["d_model"])
        self.maximum_steps = int(config["maximum_decoder_steps"])
        self.token_embedding = nn.Embedding(
            self.vocab, self.d_model, padding_idx=0
        )
        self.position_embedding = nn.Embedding(
            self.maximum_steps, self.d_model
        )
        self.embedding_dropout = nn.Dropout(float(config["dropout"]))
        layer = nn.TransformerDecoderLayer(
            d_model=self.d_model,
            nhead=int(config["nhead"]),
            dim_feedforward=int(config["dim_feedforward"]),
            dropout=float(config["dropout"]),
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.transformer_decoder = nn.TransformerDecoder(
            layer, num_layers=int(config["num_layers"])
        )
        self.final_norm = nn.LayerNorm(self.d_model)
        self.output_bias = nn.Parameter(torch.zeros(self.vocab))

    def forward(self, ids, memory, mask):
        length = ids.shape[1]
        positions = torch.arange(
            length, device=ids.device
        ).unsqueeze(0).expand(ids.shape[0], -1)
        target = (
            self.token_embedding(ids) * math.sqrt(self.d_model)
            + self.position_embedding(positions)
        )
        target = self.embedding_dropout(target)
        causal = torch.triu(
            torch.ones(
                length, length, dtype=torch.bool, device=ids.device
            ), diagonal=1
        )
        hidden = self.transformer_decoder(
            target, memory, tgt_mask=causal,
            tgt_key_padding_mask=~mask.bool(),
        )
        hidden = self.final_norm(hidden)
        return F.linear(
            hidden, self.token_embedding.weight, self.output_bias
        )

STUDENT_ENCODER = S1Encoder()
STUDENT_DECODER = SharedDecoder(decoder_config)
BASELINE_CHECKPOINT = exactly_one("s1_mobilenetv3_seed42_best.pt")
assert sha256(BASELINE_CHECKPOINT) == S1["best_checkpoint"]["sha256"]
baseline_payload = torch.load(
    BASELINE_CHECKPOINT, map_location="cpu", weights_only=False
)
assert baseline_payload["schema_version"] == (
    "phase2_s1_full_training_checkpoint_v1"
)
STUDENT_ENCODER.load_state_dict(
    baseline_payload["encoder_state_dict"], strict=True
)
STUDENT_DECODER.load_state_dict(
    baseline_payload["decoder_state_dict"], strict=True
)
BASE_ENCODER_STATE = {
    key: value.detach().cpu().clone()
    for key, value in STUDENT_ENCODER.state_dict().items()
}
BASE_DECODER_STATE = {
    key: value.detach().cpu().clone()
    for key, value in STUDENT_DECODER.state_dict().items()
}
del baseline_payload
STUDENT_ENCODER = STUDENT_ENCODER.to(device)
STUDENT_DECODER = STUDENT_DECODER.to(device)

assert sum(p.numel() for p in STUDENT_ENCODER.parameters()) == 3231024
assert sum(p.numel() for p in STUDENT_DECODER.parameters()) == 5759328
assert sum(p.numel() for p in TEACHER.parameters()) > 100_000_000



class ExpansionNetToS1Alignment(nn.Module):
    def __init__(self):
        super().__init__()
        self.projection = nn.Linear(512, 256, bias=True)
        self.norm = nn.LayerNorm(256)

    def forward(self, teacher_tokens):
        assert teacher_tokens.shape[1:] == (144, 512)
        grid = teacher_tokens.transpose(1, 2).reshape(
            teacher_tokens.shape[0], 512, 12, 12
        )
        pooled = F.adaptive_avg_pool2d(grid, (7, 7))
        tokens = pooled.flatten(2).transpose(1, 2).contiguous()
        return self.norm(self.projection(tokens))


ALIGNMENT = ExpansionNetToS1Alignment().to(device)
assert sum(parameter.numel() for parameter in ALIGNMENT.parameters()) == 131840
BASE_ALIGNMENT_STATE = {
    key: value.detach().cpu().clone()
    for key, value in ALIGNMENT.state_dict().items()
}


@torch.no_grad()
def teacher_feature_source(images):
    TEACHER.eval()
    with torch.amp.autocast("cuda", dtype=torch.float16):
        swin_tokens = TEACHER.swin_transf(images)
        assert swin_tokens.shape[1:] == (144, 1536)
        tokens = TEACHER.input_linear(swin_tokens)
    assert tokens.shape[1:] == (144, 512)
    return tokens.detach()


def normalized_feature_loss(student_tokens, aligned_teacher_tokens, eps=1e-6):
    student_unit = F.normalize(
        student_tokens.float(), p=2, dim=-1, eps=eps
    )
    teacher_unit = F.normalize(
        aligned_teacher_tokens.detach().float(), p=2, dim=-1, eps=eps
    )
    return ((student_unit - teacher_unit) ** 2).sum(dim=-1).mean()


LAMBDA_FEATURE = 0.1
BACKWARD_DIVISOR = 1024.0
ACCUMULATION = 64
EFFECTIVE_BATCH = 64
STEPS_PER_EPOCH = math.ceil(113287 / EFFECTIVE_BATCH)
MAX_EPOCHS = 20
TOTAL_STEPS = STEPS_PER_EPOCH * MAX_EPOCHS
WARMUP_STEPS = round(TOTAL_STEPS * 0.05)
PILOT_STEPS = 100
PILOT_WARMUP_STEPS = 5

backbone_parameters = list(STUDENT_ENCODER.backbone.parameters())
backbone_ids = {id(parameter) for parameter in backbone_parameters}
student_adapter_parameters = [
    parameter for parameter in STUDENT_ENCODER.parameters()
    if id(parameter) not in backbone_ids
]
alignment_parameters = list(ALIGNMENT.parameters())
adapter_parameters = student_adapter_parameters + alignment_parameters
decoder_parameters = list(STUDENT_DECODER.parameters())
all_trainable_parameters = (
    backbone_parameters + adapter_parameters + decoder_parameters
)
assert len({id(p) for p in all_trainable_parameters}) == len(
    all_trainable_parameters
)


def set_feature_train():
    STUDENT_ENCODER.train()
    STUDENT_DECODER.train()
    ALIGNMENT.train()
    batch_norm_count = 0
    for module in STUDENT_ENCODER.modules():
        if isinstance(module, nn.modules.batchnorm._BatchNorm):
            module.eval()
            batch_norm_count += 1
    assert batch_norm_count == 46


def reset_feature_pilot_state():
    STUDENT_ENCODER.load_state_dict(BASE_ENCODER_STATE, strict=True)
    STUDENT_DECODER.load_state_dict(BASE_DECODER_STATE, strict=True)
    ALIGNMENT.load_state_dict(BASE_ALIGNMENT_STATE, strict=True)
    for parameter in all_trainable_parameters:
        parameter.grad = None


def to_device(batch):
    return {
        key: (
            value.to(device, non_blocking=True)
            if isinstance(value, torch.Tensor) else value
        )
        for key, value in batch.items()
    }


def build_feature_optimizer():
    optimizer = torch.optim.AdamW(
        [
            {"params": backbone_parameters, "lr": 1e-4},
            {"params": adapter_parameters, "lr": 3e-4},
            {"params": decoder_parameters, "lr": 3e-4},
        ],
        betas=(0.9, 0.999),
        eps=1e-8,
        weight_decay=1e-4,
    )

    def factor(step):
        if step < WARMUP_STEPS:
            return (step + 1) / max(1, WARMUP_STEPS)
        progress = (
            (step - WARMUP_STEPS)
            / max(1, TOTAL_STEPS - WARMUP_STEPS)
        )
        return 0.05 + 0.95 * 0.5 * (
            1.0 + math.cos(math.pi * min(progress, 1.0))
        )

    scheduler = torch.optim.lr_scheduler.LambdaLR(
        optimizer, lr_lambda=factor
    )
    scaler = torch.amp.GradScaler(
        "cuda", enabled=True, init_scale=1024.0
    )
    return optimizer, scheduler, scaler


def feature_optimizer_step(
    microbatches, optimizer, scheduler, scaler, retry_limit=10
):
    assert 1 <= len(microbatches) <= ACCUMULATION
    human_total = sum(
        int(batch["human_ids"][:, 1:].ne(0).sum())
        for batch in microbatches
    )
    assert human_total > 0
    retries = 0
    while True:
        set_feature_train()
        optimizer.zero_grad(set_to_none=True)
        human_sum_value = 0.0
        feature_sum_value = 0.0
        for cpu_batch in microbatches:
            teacher_tokens = teacher_feature_source(
                cpu_batch["teacher_image"].to(
                    device, non_blocking=True
                )
            )
            current = to_device(cpu_batch)
            human = current["human_ids"]
            with torch.amp.autocast("cuda", dtype=torch.float16):
                student_memory = STUDENT_ENCODER(
                    current["student_image"]
                )
                human_logits = STUDENT_DECODER(
                    human[:, :-1],
                    student_memory,
                    human[:, :-1].ne(0),
                )
                aligned_teacher = ALIGNMENT(teacher_tokens)
                human_sum = F.cross_entropy(
                    human_logits.float().reshape(-1, 10080),
                    human[:, 1:].reshape(-1),
                    ignore_index=0,
                    reduction="sum",
                )
                feature_loss = normalized_feature_loss(
                    student_memory, aligned_teacher
                )
                normalized = (
                    human_sum / human_total
                    + LAMBDA_FEATURE
                    * feature_loss
                    / len(microbatches)
                )
            scaler.scale(normalized / BACKWARD_DIVISOR).backward()
            human_sum_value += float(human_sum.detach())
            feature_sum_value += float(feature_loss.detach())
            del (
                current,
                teacher_tokens,
                student_memory,
                human_logits,
                aligned_teacher,
                human_sum,
                feature_loss,
                normalized,
            )

        scaler.unscale_(optimizer)
        for parameter in all_trainable_parameters:
            if parameter.grad is not None:
                parameter.grad.mul_(BACKWARD_DIVISOR)
        gradient_norm = torch.nn.utils.clip_grad_norm_(
            all_trainable_parameters, 1.0
        )
        old_scale = scaler.get_scale()
        scaler.step(optimizer)
        scaler.update()
        new_scale = scaler.get_scale()
        if new_scale < old_scale:
            retries += 1
            assert retries <= retry_limit
            continue
        scheduler.step()
        human_ce = human_sum_value / human_total
        mean_feature = feature_sum_value / len(microbatches)
        return {
            "combined_loss": human_ce + LAMBDA_FEATURE * mean_feature,
            "human_ce": human_ce,
            "normalized_feature_loss": mean_feature,
            "human_tokens": human_total,
            "gradient_norm": float(gradient_norm),
            "overflow_retries": retries,
            "scaler": float(new_scale),
        }


assert STEPS_PER_EPOCH == 1771
assert TOTAL_STEPS == 35420
assert WARMUP_STEPS == 1771
write_json(CONFIG / "feature_kd_seed42_pilot_contract.json", {
    "schema_version": "feature_kd_seed42_pilot_contract_v1",
    "created_at_utc": now(),
    "status": "locked",
    "student_initialization": "S1 seed-42 baseline-best",
    "teacher": "ExpansionNet v2 frozen",
    "teacher_precision": "FP16 autocast, stop-gradient",
    "teacher_feature_tap": "input_linear(swin_transf(image))",
    "teacher_feature_shape": ["B", 144, 512],
    "student_feature_shape": ["B", 49, 256],
    "spatial_alignment": "adaptive average pooling 12x12 to 7x7",
    "adapter": "Linear(512,256)+LayerNorm",
    "adapter_trainable": True,
    "loss": "human CE + 0.1 * L2-normalized feature MSE",
    "physical_batch": 1,
    "gradient_accumulation": ACCUMULATION,
    "effective_batch": EFFECTIVE_BATCH,
    "steps_per_epoch": STEPS_PER_EPOCH,
    "maximum_epochs": MAX_EPOCHS,
    "full_run_optimizer_steps": TOTAL_STEPS,
    "pilot_optimizer_steps": PILOT_STEPS,
    "pilot_weights_persisted": False,
    "test_set_status": "sealed",
    "test_set_used": False,
})
print("=" * 78)
print("FEATURE-KD CELL 3 — MODEL AND PILOT ENGINE")
print("=" * 78)
print("Teacher                   : ExpansionNet v2 FROZEN")
print("Student init              : S1 SEED-42 BASELINE-BEST")
print("Alignment parameters      : 131,840")
print("Physical/accum/effective  : 1 / 64 / 64")
print("Full steps / Pilot steps  : 35,420 / 100")
print("Pilot weights persisted   : False")
print("Test Set status           : SEALED")


FEATURE-KD CELL 3 — MODEL AND PILOT ENGINE
Teacher                   : ExpansionNet v2 FROZEN
Student init              : S1 SEED-42 BASELINE-BEST
Alignment parameters      : 131,840
Physical/accum/effective  : 1 / 64 / 64
Full steps / Pilot steps  : 35,420 / 100
Pilot weights persisted   : False
Test Set status           : SEALED


## Cell 4 — Zero-update Feature-KD compatibility preflight


In [4]:
CELL_STARTED_AT = time.perf_counter()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()
reset_feature_pilot_state()

probe_iterator = iter(build_train_loader(epoch=0, cursor=0))
probe_batches = [next(probe_iterator) for _ in range(4)]
del probe_iterator

TEACHER.eval()
STUDENT_ENCODER.eval()
STUDENT_DECODER.eval()
ALIGNMENT.eval()

human_total = sum(
    int(batch["human_ids"][:, 1:].ne(0).sum())
    for batch in probe_batches
)
caption_sum = None
feature_sum = None
teacher_repeats = []
student_repeats = []
for index, cpu_batch in enumerate(probe_batches):
    teacher_images = cpu_batch["teacher_image"].to(
        device, non_blocking=True
    )
    teacher_tokens = teacher_feature_source(teacher_images)
    if index == 0:
        teacher_repeat = teacher_feature_source(teacher_images)
        teacher_repeats.append(
            float((teacher_tokens - teacher_repeat).abs().max().item())
        )
        del teacher_repeat

    current = to_device(cpu_batch)
    human = current["human_ids"]
    with torch.amp.autocast("cuda", dtype=torch.float16):
        student_memory = STUDENT_ENCODER(current["student_image"])
        if index == 0:
            student_repeat = STUDENT_ENCODER(current["student_image"])
            student_repeats.append(
                float(
                    (student_memory - student_repeat)
                    .abs().max().item()
                )
            )
            del student_repeat
        logits = STUDENT_DECODER(
            human[:, :-1], student_memory, human[:, :-1].ne(0)
        )
        aligned_teacher = ALIGNMENT(teacher_tokens)
        caption_piece = F.cross_entropy(
            logits.float().reshape(-1, 10080),
            human[:, 1:].reshape(-1),
            ignore_index=0,
            reduction="sum",
        )
        feature_piece = normalized_feature_loss(
            student_memory, aligned_teacher
        )
    caption_sum = (
        caption_piece if caption_sum is None
        else caption_sum + caption_piece
    )
    feature_sum = (
        feature_piece if feature_sum is None
        else feature_sum + feature_piece
    )

caption_loss = caption_sum / human_total
feature_loss = feature_sum / len(probe_batches)
shared_encoder_parameters = [
    parameter for parameter in STUDENT_ENCODER.parameters()
    if parameter.requires_grad
]
caption_gradients = torch.autograd.grad(
    caption_loss,
    shared_encoder_parameters,
    retain_graph=True,
    allow_unused=True,
)
feature_gradients = torch.autograd.grad(
    LAMBDA_FEATURE * feature_loss,
    shared_encoder_parameters,
    retain_graph=False,
    allow_unused=True,
)
caption_vector = torch.cat([
    gradient.detach().float().reshape(-1)
    for gradient in caption_gradients
    if gradient is not None
])
feature_vector = torch.cat([
    gradient.detach().float().reshape(-1)
    for gradient in feature_gradients
    if gradient is not None
])
caption_gradient_norm = float(
    torch.linalg.vector_norm(caption_vector).item()
)
feature_gradient_norm = float(
    torch.linalg.vector_norm(feature_vector).item()
)
gradient_ratio = feature_gradient_norm / max(
    caption_gradient_norm, 1e-12
)
finite = bool(
    torch.isfinite(caption_loss).item()
    and torch.isfinite(feature_loss).item()
    and torch.isfinite(caption_vector).all().item()
    and torch.isfinite(feature_vector).all().item()
)
repeat_exact = bool(
    max(teacher_repeats + student_repeats) == 0.0
)
peak_memory_mib = torch.cuda.max_memory_allocated() / (1024 ** 2)
ratio_min, ratio_max = [
    float(value)
    for value in FEATURE_CONTRACT["preflight_gates"][
        "weighted_feature_to_caption_gradient_ratio"
    ]
]
ratio_passed = bool(
    finite
    and caption_gradient_norm > 0.0
    and feature_gradient_norm > 0.0
    and ratio_min <= gradient_ratio <= ratio_max
)
memory_passed = bool(
    peak_memory_mib
    < float(
        FEATURE_CONTRACT["preflight_gates"][
            "t4_peak_memory_mib_less_than"
        ]
    )
)
teacher_gradient_tensors = sum(
    parameter.grad is not None for parameter in TEACHER.parameters()
)
PREFLIGHT_PASSED = bool(
    ratio_passed
    and memory_passed
    and finite
    and repeat_exact
    and teacher_gradient_tensors == 0
)

preflight_payload = {
    "schema_version": "feature_kd_seed42_preflight_v1",
    "created_at_utc": now(),
    "status": "passed" if PREFLIGHT_PASSED else "failed",
    "student_initialization": "S1 seed-42 baseline-best",
    "probe_train_images": len(probe_batches),
    "caption_ce": float(caption_loss.detach()),
    "normalized_feature_loss": float(feature_loss.detach()),
    "lambda_feature": LAMBDA_FEATURE,
    "caption_encoder_gradient_norm": caption_gradient_norm,
    "weighted_feature_encoder_gradient_norm": feature_gradient_norm,
    "weighted_feature_to_caption_gradient_ratio": gradient_ratio,
    "accepted_gradient_ratio_interval": [ratio_min, ratio_max],
    "gradient_ratio_gate_passed": ratio_passed,
    "finite_loss_and_gradients": finite,
    "deterministic_repeat": repeat_exact,
    "peak_gpu_memory_mib": peak_memory_mib,
    "memory_gate_passed": memory_passed,
    "teacher_gradient_tensors": teacher_gradient_tensors,
    "optimizer_steps": 0,
    "full_training_started": False,
    "test_set_status": "sealed",
    "test_set_used": False,
    "elapsed_seconds": time.perf_counter() - CELL_STARTED_AT,
}
write_json(
    RESULT / "feature_kd_seed42_preflight_result.json",
    preflight_payload,
)
write_json(
    AUDIT / "feature_kd_cell04_zero_update_preflight.json",
    preflight_payload,
)

del (
    caption_gradients,
    feature_gradients,
    caption_vector,
    feature_vector,
    caption_sum,
    feature_sum,
    caption_loss,
    feature_loss,
    probe_batches,
)
reset_feature_pilot_state()
gc.collect()
torch.cuda.empty_cache()

print("=" * 78)
print("FEATURE-KD CELL 4 — ZERO-UPDATE PREFLIGHT")
print("=" * 78)
print("Status                    :", preflight_payload["status"].upper())
print("Student init              : S1 seed-42 BASELINE-BEST")
print("Gradient ratio            :", gradient_ratio)
print("Accepted interval         :", [ratio_min, ratio_max])
print("Peak GPU memory MiB       :", peak_memory_mib)
print("Deterministic repeat      :", repeat_exact)
print("Optimizer steps           : 0")
print("Test Set status           : SEALED")


FEATURE-KD CELL 4 — ZERO-UPDATE PREFLIGHT
Status                    : FAILED
Student init              : S1 seed-42 BASELINE-BEST
Gradient ratio            : 0.02336522678514595
Accepted interval         : [0.05, 0.5]
Peak GPU memory MiB       : 1188.44970703125
Deterministic repeat      : True
Optimizer steps           : 0
Test Set status           : SEALED


## Cell 5 — Discarded 100-step ETA pilot


In [5]:
CELL_STARTED_AT = time.perf_counter()
PILOT_RAN = False
PILOT_PASSED = False
pilot_summary = None

if not PREFLIGHT_PASSED:
    print("=" * 78)
    print("FEATURE-KD CELL 5 — 100-STEP ETA PILOT")
    print("=" * 78)
    print("Status                    : BLOCKED")
    print("Reason                    : zero-update preflight failed")
    print("Optimizer steps           : 0")
    print("Pilot weights persisted   : False")
else:
    reset_feature_pilot_state()
    optimizer, scheduler, scaler = build_feature_optimizer()
    loader_iterator = iter(build_train_loader(epoch=0, cursor=0))
    step_records = []
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

    for pilot_step in range(1, PILOT_STEPS + 1):
        torch.cuda.synchronize()
        step_started = time.perf_counter()
        microbatches = [
            next(loader_iterator) for _ in range(ACCUMULATION)
        ]
        result = feature_optimizer_step(
            microbatches, optimizer, scheduler, scaler
        )
        torch.cuda.synchronize()
        elapsed = time.perf_counter() - step_started
        record = {
            "pilot_step": pilot_step,
            "elapsed_seconds": elapsed,
            **result,
            "peak_gpu_memory_mib": (
                torch.cuda.max_memory_allocated() / (1024 ** 2)
            ),
        }
        step_records.append(record)
        if pilot_step == 1 or pilot_step % 10 == 0:
            print(
                f"Pilot {pilot_step:03d}/{PILOT_STEPS}: "
                f"{elapsed:.3f} s/step | "
                f"loss={result['combined_loss']:.6f} | "
                f"peak={record['peak_gpu_memory_mib']:.1f} MiB"
            )
        del microbatches, result

    del loader_iterator
    PILOT_RAN = True
    measured = np.asarray(
        [
            row["elapsed_seconds"]
            for row in step_records[PILOT_WARMUP_STEPS:]
        ],
        dtype=np.float64,
    )
    median_seconds = float(np.median(measured))
    p90_seconds = float(np.percentile(measured, 90))
    p95_seconds = float(np.percentile(measured, 95))
    mean_seconds = float(np.mean(measured))
    peak_memory_mib = float(max(
        row["peak_gpu_memory_mib"] for row in step_records
    ))
    overflow_retries = int(sum(
        row["overflow_retries"] for row in step_records
    ))
    stability_ratio = p95_seconds / max(median_seconds, 1e-12)

    # Step-only ETA plus an explicitly disclosed 2-5 h allowance for
    # validation, packaging and session startup over a 20-epoch run.
    calendar_multiplier = 1.284
    eta_rows = []
    for epochs in [13, 15, 18, 20]:
        steps = STEPS_PER_EPOCH * epochs
        step_hours_median = median_seconds * steps / 3600.0
        step_hours_p95 = p95_seconds * steps / 3600.0
        active_low = step_hours_median + 2.0
        active_high = step_hours_p95 + 5.0
        eta_rows.append({
            "epochs": epochs,
            "optimizer_steps": steps,
            "step_only_hours_median": step_hours_median,
            "step_only_hours_p95": step_hours_p95,
            "active_gpu_hours_low": active_low,
            "active_gpu_hours_high": active_high,
            "calendar_days_low": (
                active_low * calendar_multiplier / 24.0
            ),
            "calendar_days_high": (
                active_high * calendar_multiplier / 24.0
            ),
        })

    pilot_state_changed = any(
        not torch.equal(
            STUDENT_ENCODER.state_dict()[key].detach().cpu(), expected
        )
        for key, expected in BASE_ENCODER_STATE.items()
    ) or any(
        not torch.equal(
            STUDENT_DECODER.state_dict()[key].detach().cpu(), expected
        )
        for key, expected in BASE_DECODER_STATE.items()
    ) or any(
        not torch.equal(
            ALIGNMENT.state_dict()[key].detach().cpu(), expected
        )
        for key, expected in BASE_ALIGNMENT_STATE.items()
    )

    state_restored = True
    reset_feature_pilot_state()
    for key, expected in BASE_ENCODER_STATE.items():
        state_restored &= torch.equal(
            STUDENT_ENCODER.state_dict()[key].detach().cpu(), expected
        )
    for key, expected in BASE_DECODER_STATE.items():
        state_restored &= torch.equal(
            STUDENT_DECODER.state_dict()[key].detach().cpu(), expected
        )
    for key, expected in BASE_ALIGNMENT_STATE.items():
        state_restored &= torch.equal(
            ALIGNMENT.state_dict()[key].detach().cpu(), expected
        )

    PILOT_PASSED = bool(
        len(step_records) == PILOT_STEPS
        and np.isfinite(measured).all()
        and overflow_retries == 0
        and stability_ratio <= 1.50
        and peak_memory_mib < 14500.0
        and pilot_state_changed
        and state_restored
    )
    pilot_summary = {
        "schema_version": "feature_kd_seed42_eta_pilot_v1",
        "created_at_utc": now(),
        "status": "passed" if PILOT_PASSED else "failed",
        "student_initialization": "S1 seed-42 baseline-best",
        "optimizer_steps": PILOT_STEPS,
        "warmup_steps_excluded_from_eta": PILOT_WARMUP_STEPS,
        "timed_steps": int(measured.size),
        "seconds_per_optimizer_step": {
            "mean": mean_seconds,
            "median": median_seconds,
            "p90": p90_seconds,
            "p95": p95_seconds,
        },
        "p95_to_median_ratio": stability_ratio,
        "peak_gpu_memory_mib": peak_memory_mib,
        "overflow_retries": overflow_retries,
        "eta": eta_rows,
        "calendar_multiplier": calendar_multiplier,
        "calendar_multiplier_source": (
            "observed seed-42 Dense-KD calendar/active ratio"
        ),
        "validation_packaging_allowance_hours": [2.0, 5.0],
        "pilot_state_changed_before_restore": bool(pilot_state_changed),
        "pilot_state_restored_exact": bool(state_restored),
        "pilot_weights_persisted": False,
        "full_training_started": False,
        "test_set_status": "sealed",
        "test_set_used": False,
        "elapsed_seconds": time.perf_counter() - CELL_STARTED_AT,
    }
    write_json(
        RESULT / "feature_kd_seed42_eta_pilot.json",
        pilot_summary,
    )
    STEP_LOG = LOG / "feature_kd_seed42_eta_pilot_steps.jsonl"
    with STEP_LOG.open("w", encoding="utf-8") as handle:
        for row in step_records:
            handle.write(json.dumps(row, ensure_ascii=False) + "\n")
    write_json(
        AUDIT / "feature_kd_cell05_eta_pilot.json",
        pilot_summary,
    )

    del optimizer, scheduler, scaler, step_records, measured
    gc.collect()
    torch.cuda.empty_cache()

    eta20 = eta_rows[-1]
    print("=" * 78)
    print("FEATURE-KD CELL 5 — 100-STEP ETA PILOT")
    print("=" * 78)
    print("Status                    :", pilot_summary["status"].upper())
    print("Optimizer steps           :", PILOT_STEPS)
    print("Median / p95 s/step       :", median_seconds, "/", p95_seconds)
    print("Peak GPU memory MiB       :", peak_memory_mib)
    print("Overflow retries          :", overflow_retries)
    print("Pilot changed state       :", pilot_state_changed)
    print(
        "20-epoch active GPU hours :",
        round(eta20["active_gpu_hours_low"], 2),
        "-",
        round(eta20["active_gpu_hours_high"], 2),
    )
    print(
        "20-epoch calendar days    :",
        round(eta20["calendar_days_low"], 2),
        "-",
        round(eta20["calendar_days_high"], 2),
    )
    print("Pilot state restored      :", state_restored)
    print("Pilot weights persisted   : False")
    print("Test Set status           : SEALED")


FEATURE-KD CELL 5 — 100-STEP ETA PILOT
Status                    : BLOCKED
Reason                    : zero-update preflight failed
Optimizer steps           : 0
Pilot weights persisted   : False


## Cell 6 — Package, manifest và quyết định


In [6]:
CELL_STARTED_AT = time.perf_counter()
import csv

FINAL_STATUS = (
    "authorized"
    if PREFLIGHT_PASSED and PILOT_PASSED
    else "blocked"
)
decision = {
    "schema_version": "feature_kd_seed42_pilot_decision_v1",
    "created_at_utc": now(),
    "status": FINAL_STATUS,
    "preflight_passed": PREFLIGHT_PASSED,
    "eta_pilot_ran": PILOT_RAN,
    "eta_pilot_passed": PILOT_PASSED,
    "student_initialization": "S1 seed-42 baseline-best",
    "authorized_next": (
        "Feature-KD seed-42 screening full run"
        if FINAL_STATUS == "authorized"
        else "Stop Feature-KD; inspect failed gate before any training"
    ),
    "confirmation_seeds_authorized": False,
    "pilot_weights_persisted": False,
    "full_training_started": False,
    "selection_split": "validation",
    "test_set_status": "sealed",
    "test_set_used": False,
}
write_json(RESULT / "feature_kd_seed42_pilot_decision.json", decision)

marker_name = (
    "FEATURE_KD_SEED42_FULL_RUN_AUTHORIZED.json"
    if FINAL_STATUS == "authorized"
    else "FEATURE_KD_SEED42_FULL_RUN_BLOCKED.json"
)
MARKER_PATH = OUT / marker_name
write_json(MARKER_PATH, decision)

artifact_paths = []
for directory in [AUDIT, CONFIG, LOG, RESULT]:
    artifact_paths.extend(
        path for path in directory.rglob("*") if path.is_file()
    )
artifact_paths.append(MARKER_PATH)
artifact_paths = sorted(set(artifact_paths))

MANIFEST_CSV = MANIFEST / "feature_kd_seed42_pilot_manifest.csv"
with MANIFEST_CSV.open("w", encoding="utf-8", newline="") as handle:
    writer = csv.DictWriter(
        handle,
        fieldnames=["relative_path", "size_bytes", "sha256"],
    )
    writer.writeheader()
    for path in artifact_paths:
        writer.writerow({
            "relative_path": path.relative_to(OUT).as_posix(),
            "size_bytes": path.stat().st_size,
            "sha256": sha256(path),
        })

with MANIFEST_CSV.open("r", encoding="utf-8", newline="") as handle:
    manifest_rows = list(csv.DictReader(handle))
for row in manifest_rows:
    artifact = resolve_inside(OUT, row["relative_path"])
    assert artifact.stat().st_size == int(row["size_bytes"])
    assert sha256(artifact) == row["sha256"]

complete = {
    **decision,
    "schema_version": "feature_kd_seed42_preflight_eta_complete_v1",
    "manifest": {
        "relative_path": MANIFEST_CSV.relative_to(OUT).as_posix(),
        "sha256": sha256(MANIFEST_CSV),
        "rows": len(manifest_rows),
    },
    "elapsed_seconds": time.perf_counter() - CELL_STARTED_AT,
}
COMPLETE_PATH = (
    OUT / "XLKD05_Dense_Mapped_LogitKD_"
    "FEATURE_KD_SEED42_PREFLIGHT_ETA_COMPLETE.json"
)
write_json(COMPLETE_PATH, complete)

print("=" * 78)
print("FEATURE-KD CELL 6 — PACKAGE DECISION")
print("=" * 78)
print("Status                    :", FINAL_STATUS.upper())
print("Preflight                 :", PREFLIGHT_PASSED)
print("100-step ETA pilot        :", PILOT_PASSED)
print("Manifest rows             :", len(manifest_rows))
print("Manifest verification     : ALL PASSED")
print("Pilot weights persisted   : False")
print("Full training started     : False")
print("Confirmation seeds        : NOT AUTHORIZED")
print("Test Set status           : SEALED")
print("Marker                    :", MARKER_PATH)
print("Complete                  :", COMPLETE_PATH)


FEATURE-KD CELL 6 — PACKAGE DECISION
Status                    : BLOCKED
Preflight                 : False
100-step ETA pilot        : False
Manifest rows             : 10
Manifest verification     : ALL PASSED
Pilot weights persisted   : False
Full training started     : False
Confirmation seeds        : NOT AUTHORIZED
Test Set status           : SEALED
Marker                    : /kaggle/working//XLKD05_Dense_Mapped_LogitKD_feature_kd_seed42_pilot/FEATURE_KD_SEED42_FULL_RUN_BLOCKED.json
Complete                  : /kaggle/working//XLKD05_Dense_Mapped_LogitKD_feature_kd_seed42_pilot/XLKD05_Dense_Mapped_LogitKD_FEATURE_KD_SEED42_PREFLIGHT_ETA_COMPLETE.json
